# Data Extraction
Aug 16, 2022

Yuhan Ye

References: 
https://marcosammon.com/teaching/
https://sraf.nd.edu/textual-analysis/

## Basic tools: Regular Expressions

In [1]:
#Import Python Packages
import numpy as np
import pandas as pd
import os
import re
from bs4 import BeautifulSoup
import codecs
import csv
import string
import nltk

In [2]:
nltk.download('punkt') #Now updated for Python 3.X

[nltk_data] Downloading package punkt to /Users/f0034w9/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
test="101000000000100"
#* is greedy, match 0 or more
#*? is reluctant, match zero or 1 (and no more)

In [4]:
t2=re.findall("1.*1",test)
print(t2)

['1010000000001']


In [5]:
t3=re.findall("1.*?1",test) #stops at the 3rd digit and no more
print(t3)

['101']


In [9]:
# using finditer to count words
string="better ingredients, better pizza, papa john's pizza"
nummatch = sum(1 for _ in re.finditer(r'\bpizza\b',
                                      string, flags=re.IGNORECASE))
print(nummatch)

3


## Basic tools:  Wordstems

In [10]:
# Create p_stemmer of class PorterStemmer
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import RegexpTokenizer

In [11]:
wordtest="regulation regulatory regulations"
p_stemmer = PorterStemmer()
tokenizer = RegexpTokenizer(r'\w+')

In [12]:
print("string: ", wordtest)

string:  regulation regulatory regulations


In [13]:
tokens = tokenizer.tokenize(wordtest)
print("tokens: ",tokens)

tokens:  ['regulation', 'regulatory', 'regulations']


In [14]:
stemmed_tokens = [p_stemmer.stem(i) for i in tokens]
stemmedcorp = " ".join(stemmed_tokens)
print("stemmed tokens:", stemmedcorp)

stemmed tokens: regul regulatori regul


## Basic tools:  Section Extracts

In [16]:
#String
ftext="item1a. section to extract item1b section to ignore"

In [17]:
#What to find
regexTxt = 'item[^a-zA-Z\n]*1a\..*?item[^a-zA-Z\n]*1b'

In [22]:
#Note, this will include both "bookends"
section = re.findall(regexTxt, ftext, re.IGNORECASE | re.DOTALL)

In [23]:
section

['item1a. section to extract item1b']

In [28]:
#re.findall returns a list -- this converts it to a string
section=section[0]

In [29]:
section

'item1a. section to extract item1b'

In [30]:
#Remove the bookends with re.sub
section = re.sub('item[^a-zA-Z\n]*1a\.',"",section,flags=re.IGNORECASE)
section = re.sub('item[^a-zA-Z\n]*1b',"",section,flags=re.IGNORECASE)
print(section)

 section to extract 


## Now, let's look into a single 10-K file as an example

In [31]:
#Regular expressions we will use to identify the business description section
#The rbusiness description section is item 1, so we want to identify the section
#between item 1 and item 1a, or item 1 and item 1b
regexTxt = 'item[^a-zA-Z\n]*1\..*?item[^a-zA-Z\n]*1a'
regexTxt2 = 'item[^a-zA-Z\n]*1\..*?item[^a-zA^Z\n]*1b'
regexTxtEx = 'item[^a-zA-Z\n]*1.*?item[^a-zA^Z\n]*1a'
regexTxtEx2 = 'item[^a-zA-Z\n]*1.*?item[^a-zA^Z\n]*1b'

In [32]:
#Set parset for beautiful soup
paser = 'html.parser'
#paser = 'html5lib'

In [33]:
os.chdir('/Users/f0034w9/Dropbox (Dartmouth College)/Exercise Computational Linguistics/data/')

In [34]:
#Read the Access Power Inc 2021 Q1 10K file
with codecs.open('AccessPower10K.txt', 'r', encoding='utf8', errors='replace') as myfile:
    ftext = myfile.read().replace('\n', ' ')

In [35]:
#Use beautiful soup to clean the file if it is html
if '<html>' in ftext.lower():
  try:
      ftext = BeautifulSoup(ftext,paser).get_text()
  except Exception as e:
      print(str(e))

In [36]:
#first pass at identifying the business description section
section = re.findall('item[^a-zA-Z\n]*1.*item[^a-zA-Z\n]*1a',ftext, re.IGNORECASE | re.DOTALL)

In [37]:
section

["ITEM 1.   DESCRIPTION OF BUSINESS  Access-Power & Co., Inc, is a for profit business looking for a MERGER CANDIDATE.  We believe that a REVERSE MERGER candidate is ready to happen in 2021.  2020 was a bad year with the Biblical Covid-19 attack on mankind. We believe this aggression will continue into 2021, and affect consumer demand.  THE COMPANY HAS NOT HAD ANY CONSISTENT REVENUE SINCE MAY 2020.  The Company struggled from May 1, 2019 to October 18, 2019.  We had no income during this period, and our operational expenses were paid for by myself,  Patrick J. Jensen as a donation to the Company.  I personally paid out of my own pocket all the expenses during this dark time, EVEN TODAY.  On October 2, 2019--->  I dreamed of getting off the greys.   This is a fact.  The Company also operates many eCommerce websites at the present time.  https://www.clonesbydrones.com https://www.clonesbycar.com (s) https://www.mycbdpets.com  We continue to strive to build up our revenues. We want to suc

In [38]:
#If that fails, try alternative 1
if len(section) ==0 :   
  section = re.findall(regexTxtEx,ftext, re.IGNORECASE | re.DOTALL)
  #If alternative 1 fails, try alternative 2
  if len(section) ==0 :
      section = re.findall(regexTxtEx2,ftext, re.IGNORECASE | re.DOTALL)
#If it succeeds, extract the business description section
else:
  section = re.findall(regexTxt,ftext, re.IGNORECASE | re.DOTALL)
  if len(section) ==0:
      section = re.findall(regexTxt2,ftext, re.IGNORECASE | re.DOTALL)

In [39]:
#Sometimes there will be multiple matches
#A common case is one match in the table of contents, and one match
#in the body of the document.
#Taking the longer section avoids the mistaken match to the
#table of contents.
#print('find ' + str(len(section)) + ' instance(s) of item 1A')

result = max(section,key=len)   #take the longer section 
### try to take the 2nd section?

In [40]:
result #now we see full text of the business description section in 10-K

"ITEM 1.   DESCRIPTION OF BUSINESS  Access-Power & Co., Inc, is a for profit business looking for a MERGER CANDIDATE.  We believe that a REVERSE MERGER candidate is ready to happen in 2021.  2020 was a bad year with the Biblical Covid-19 attack on mankind. We believe this aggression will continue into 2021, and affect consumer demand.  THE COMPANY HAS NOT HAD ANY CONSISTENT REVENUE SINCE MAY 2020.  The Company struggled from May 1, 2019 to October 18, 2019.  We had no income during this period, and our operational expenses were paid for by myself,  Patrick J. Jensen as a donation to the Company.  I personally paid out of my own pocket all the expenses during this dark time, EVEN TODAY.  On October 2, 2019--->  I dreamed of getting off the greys.   This is a fact.  The Company also operates many eCommerce websites at the present time.  https://www.clonesbydrones.com https://www.clonesbycar.com (s) https://www.mycbdpets.com  We continue to strive to build up our revenues. We want to succ

In [41]:
#Remove extra spaces and string "table of contents", which may appear
#on every page
result = re.sub(r'table of contents', ' ', result, flags=re.IGNORECASE)
result = result.strip()
result = re.sub('\s+', ' ', result).strip()
# \S+ means “a string of non-whitespace characters” 
# \s+ means “a string of whitespace characters”

In [42]:
result

"ITEM 1. DESCRIPTION OF BUSINESS Access-Power & Co., Inc, is a for profit business looking for a MERGER CANDIDATE. We believe that a REVERSE MERGER candidate is ready to happen in 2021. 2020 was a bad year with the Biblical Covid-19 attack on mankind. We believe this aggression will continue into 2021, and affect consumer demand. THE COMPANY HAS NOT HAD ANY CONSISTENT REVENUE SINCE MAY 2020. The Company struggled from May 1, 2019 to October 18, 2019. We had no income during this period, and our operational expenses were paid for by myself, Patrick J. Jensen as a donation to the Company. I personally paid out of my own pocket all the expenses during this dark time, EVEN TODAY. On October 2, 2019---> I dreamed of getting off the greys. This is a fact. The Company also operates many eCommerce websites at the present time. https://www.clonesbydrones.com https://www.clonesbycar.com (s) https://www.mycbdpets.com We continue to strive to build up our revenues. We want to succeed and we will c

In [43]:
#Write business description section to a text file
with codecs.open('AccessPower10K2.txt', 'w', 'utf-8') as g:
    g.write(result)
    g.close()

Now we can see the difference of the two .txt files in the directory

In [44]:
#\b - word boundary (a word is a sequence of word characters)
#\w - word characters, usually [a-zA-Z0-9_]
# * - match 0 or more of the preceding expression
# This finds any word containing "market"
marketwords = re.findall(r'\b\w*market\w*\b', result, flags=re.IGNORECASE)

In [45]:
marketwords

['market',
 'market',
 'market',
 'Markets',
 'MARKETS',
 'Markets',
 'Markets',
 'Markets',
 'Markets',
 'otcmarkets',
 'Markets',
 'Markets',
 'Markets',
 'Markets']

In [46]:
print("Unique Words Containing \"market\"")
print(set(marketwords))

Unique Words Containing "market"
{'MARKETS', 'Markets', 'market', 'otcmarkets'}


In [47]:
print("Number of Words Containing \"market\": ",len(marketwords))

Number of Words Containing "market":  14


In [48]:
wordcount = len(result.split())
print("Total Number of Words: ",wordcount)

Total Number of Words:  832


## Next, we clean multiple 10-K files at a time

In [49]:
#Regular expressions we will use to identify the business description section
#The rbusiness description section is item 1, so we want to identify the section
#between item 1 and item 1a, or item 1 and item 1b
regexTxt = 'item[^a-zA-Z\n]*1\..*?item[^a-zA-Z\n]*1a'
regexTxt2 = 'item[^a-zA-Z\n]*1\..*?item[^a-zA^Z\n]*1b'
regexTxtEx = 'item[^a-zA-Z\n]*1.*?item[^a-zA^Z\n]*1a'
regexTxtEx2 = 'item[^a-zA-Z\n]*1.*?item[^a-zA^Z\n]*1b'


In [50]:
paser = 'html.parser'
# paser = 'html5lib'
max_attempt = 3

In [51]:
pwd

'/Users/f0034w9/Dropbox (Dartmouth College)/Exercise Computational Linguistics/data'

In [53]:
os.chdir('./rantexts')

In [52]:
pwd

'/Users/f0034w9/Dropbox (Dartmouth College)/Exercise Computational Linguistics/data'

In [58]:
path_of_the_directory='/Users/yuhanye/Desktop/rantexts'

In [44]:
# print("Files and directories in a specified path:")
# for filename in os.listdir(path_of_the_directory):
#     f = os.path.join(path_of_the_directory,filename)
#     if os.path.isfile(f):
#         print(f)   ##check the file names in the folder to make sure

In [58]:
path_of_the_directory ='/Users/f0034w9/Dropbox (Dartmouth College)/Exercise Computational Linguistics/data/rantexts'

In [59]:
######%%%%%%%still need to solve: how to remove tables ########%%%%%%%%%%%
######%%%sometimes the code extracts from item10 in tables and end at item 1a in body#%%%%

#load the 10-K files saved in local drive and clean the data
for filename in os.listdir(path_of_the_directory): ###need to set directory 
#     filename ='/Users/yuhanye/Desktop/subtexts/2018\QTR1\20180227_10-K_edgar_data_1196501_0001171843-18-001503_1.txt'                                               ###to the data to run this
    with codecs.open(filename, 'r', encoding='utf8', errors='replace') as myfile:
        ftext = myfile.read().replace('\n', ' ')
    
    
    ####%%%%%%%%%%% remove html part  ####%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    
    #Use beautiful soup to clean the file if it is html
    if '<html>' in ftext.lower(): 
        try:
            ftext = BeautifulSoup(ftext,paser).get_text()
            #print(ftext)
        except Exception as e:
            print(str(e))
            
    ####%%%%%%% extract business description item 1 ####%%%%%%%%%%%%%%
    
    #first pass at identifying the business description section
    section = re.findall('item[^a-zA-Z\n]*1.*item[^a-zA-Z\n]*1a',ftext, re.IGNORECASE | re.DOTALL)
    #print(section)   
    
    #If that fails, try alternative 1
    if len(section) ==0:
        section = re.findall(regexTxtEx,ftext, re.IGNORECASE | re.DOTALL)
        #If alternative 1 fails, try alternative 2
        if len(section) ==0 :
            section = re.findall(regexTxtEx2,ftext, re.IGNORECASE | re.DOTALL)
    
    #If it succeeds, extract the business description section
    else:
        section = re.findall(regexTxt,ftext, re.IGNORECASE | re.DOTALL)
        if len(section) ==0:
            section = re.findall(regexTxt2,ftext, re.IGNORECASE | re.DOTALL)
    #print(section)
    
    #Sometimes there will be multiple matches
    #A common case is one match in the table of contents, and one match
    #in the body of the document.
    #Taking the longer section avoids the mistaken match to the
    #table of contents.
    
    #print('find ' + str(len(section)) + ' instance(s) of item 1')
    
    result = max(section,key=len, default=None) #Taking the longer section
    #print(result)  #display the content of item 1 business descriptions
    
    #Write business description section to a text file
    # save the extracted item 1 information in the "subtextsout" folder
   
    try:
        if len(result.encode('utf-8')) > 2000:  #only save string that is longer than 2kb
            #show example of .txt that start from item 11 outline
            with codecs.open('./temp_out/'+ filename, 'w', 'utf-8') as g:
                g.write(result)
                g.close()
    except:
        pass 

/Users/f0034w9/opt/anaconda3/lib/python3.7/site-packages/bs4/builder/_htmlparser.py:78: UserWarning: unknown status keyword 'D' in marked section
  warnings.warn(msg)


local variable 'match' referenced before assignment


In [61]:
# you can see: out of 61 sample files, only 56 are saved and 5 are deleted 
## as they are smaller than 2kb